In [0]:
base_path = "/Volumes/nyc/default/"
yellow_path = base_path + "yellowtaxi/"

In [0]:
from pyspark.sql.types import StringType, LongType, IntegerType, DateType, StructType, StructField, DoubleType, TimestampType

yellowSchema = StructType([
    StructField("id", StringType(), True), 
    StructField("vendorID", IntegerType(), True), 
    StructField("pickupDatetime", TimestampType(), True), 
    StructField("dropoffDatetime", TimestampType(), True), 
    StructField("passengerCount", IntegerType(), True), 
    StructField("pickupLongitude", DoubleType(), True), 
    StructField("pickupLatitude", DoubleType(), True), 
    StructField("dropoffLongitude", DoubleType(), True), 
    StructField("dropoffLatitude", DoubleType(), True), 
    StructField("storeAndFwdFlag", StringType(), True), 
    StructField("tripDuration", IntegerType(), True)
])

yellow_path = "/Volumes/nyc/default/yellowtaxilola/train.csv"

yellow_taxi_kaggle_df = (
    spark.read
    .format("csv")
    .schema(yellowSchema)
    .option("header", "true")
    .load(yellow_path)
) 

In [0]:
%pip install h3

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import h3 

# 1. 注册 UDF
@udf(returnType=StringType())
def get_h3_spark(lat, lng):
    if lat is not None and lng is not None:
        return h3.latlng_to_cell(lat, lng, 8) # 这里固定分辨率为 8
    return None

# 2. 直接对千万级 Spark DataFrame 进行套用
yellow_taxi_enriched_kaggle_df = yellow_taxi_kaggle_df.withColumn(
    "pickup_actual_h3", 
    get_h3_spark("pickupLatitude", "pickupLongitude")
)

yellow_taxi_enriched_kaggle_df.select("pickupLatitude", "pickupLongitude", "pickup_actual_h3").show(5)

In [0]:
%pip install folium

In [0]:
import folium
from folium.plugins import HeatMap

# 1. 准备数据：确保字段名完全匹配（Kaggle 数据通常是 pickup_latitude/longitude）
# 加上过滤逻辑，防止空值破坏地图
plot_data = yellow_taxi_enriched_kaggle_df.select("pickupLatitude", "pickupLongitude") \
    .dropna() \
    .limit(2000) \
    .toPandas()

# 2. 创建地图：使用 'CartoDB positron'，因为它更柔和，能突出热力图的颜色
m = folium.Map(location=[40.7306, -73.9352], zoom_start=12, tiles='CartoDB positron')

# 3. 添加热力图
heat_values = plot_data[['pickupLatitude', 'pickupLongitude']].values.tolist()
HeatMap(heat_values, radius=10, blur=15).add_to(m)

# 4. 显示地图
m

In [0]:
import folium
from folium.plugins import HeatMap

# 1. 提高采样量 (增加到 10000 条，只要浏览器显存够，这会更精细)
plot_data_fine = yellow_taxi_enriched_kaggle_df.select("pickupLatitude", "pickupLongitude") \
    .dropna() \
    .limit(10000) \
    .toPandas()

# 2. 创建底图
m_fine = folium.Map(location=[40.7306, -73.9352], zoom_start=14, tiles='CartoDB positron')

# 3. 配置精细参数
# radius 调小（5-8），blur 调小（8-10），增加 min_opacity 确保孤立点也可见
heat_values_fine = plot_data_fine[['pickupLatitude', 'pickupLongitude']].values.tolist()
HeatMap(
    heat_values_fine, 
    radius=7, 
    blur=8, 
    min_opacity=0.4,
    gradient={0.4: 'blue', 0.65: 'lime', 1: 'red'} # 自定义渐变色，增强对比度
).add_to(m_fine)

m_fine

In [0]:
import folium
import h3
import pandas as pd
import numpy as np

# 1. 在 Spark 中进行聚合（这是 DE 的核心工作）
h3_counts = yellow_taxi_enriched_kaggle_df.groupBy("pickup_h3").count().toPandas()

# 2. 准备颜色映射 (使用分位数确保颜色分布均匀)
# 剔除异常值后，将 count 映射到 0-1 之间
max_count = h3_counts['count'].quantile(0.95)
h3_counts['score'] = h3_counts['count'] / max_count

# 3. 定义颜色函数
def get_color(score):
    # 从黄色(低)到红色(高)
    if score > 0.8: return '#800026' # 深红
    if score > 0.5: return '#BD0026'
    if score > 0.2: return '#FC4E2A'
    return '#FFEDA0' # 浅黄

# 4. 绘图
m_h3 = folium.Map(location=[40.7306, -73.9352], zoom_start=12, tiles='CartoDB dark_matter')

for _, row in h3_counts.head(1000).iterrows(): # 绘制前1000个最火爆的格子
    hex_id = row['pickup_h3']
    if hex_id:
        points = h3.cell_to_boundary(hex_id)
        folium.Polygon(
            locations=points,
            fill=True,
            fill_color=get_color(row['score']),
            color=None, # 不显示边框线，更美观
            fill_opacity=0.6,
            popup=f"H3_ID: {hex_id}<br>订单数: {row['count']}"
        ).add_to(m_h3)

m_h3

# Analyze h3 location identified by PULocation 

In [0]:
df = (spark
      .read
      .parquet("/Volumes/nyc/default/yellowtaxi/yellow_tripdata_2016-01.parquet"))

ddl_schema = df.schema.toDDL()

print(ddl_schema)

In [0]:
from pyspark.sql.types import StringType, LongType, IntegerType, DateType, StructType, StructField, DoubleType, TimestampType

base_path = "/Volumes/nyc/default/"
yellow_path = base_path + "yellowtaxi/" 


yellowSchema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True)
    #,
    #StructField("cbd_congestion_fee", DoubleType(), True)
])


yellow_taxi_raw_df = (
    spark.read
    .format("parquet")
    .schema(yellowSchema)
    .load(yellow_path)
)

In [0]:
from pyspark.sql.functions import expr 

a = yellow_taxi_df.withColumn("YYYYMM", expr("year(tpep_dropoff_datetime)*100 + month(tpep_dropoff_datetime)")).groupBy("YYYYMM").count().orderBy("YYYYMM")

a.display()

### Generate h3 for PULocation

In [0]:
%pip install geopandas

In [0]:
import h3
import geopandas as gpd
import folium
from shapely.geometry import mapping

# 1. 加载并转换坐标系 (确保列名匹配你的 Index: zone, LocationID, borough)
gdf = gpd.read_file("/Volumes/nyc/default/nyczone/taxi_zones/taxi_zones.shp").to_crs("EPSG:4326")

# 2. 定义 H3 转换函数 (使用你之前调通的质心方案，最稳健)
def get_h3_cell(geom, res=8):
    centroid = geom.centroid
    try:
        # 兼容 H3 4.x 和 3.x
        return h3.latlng_to_cell(centroid.y, centroid.x, res) if hasattr(h3, 'latlng_to_cell') else h3.geo_to_h3(centroid.y, centroid.x, res)
    except:
        return None

# 3. 生成数据
gdf['centroid_h3_cells'] = gdf['geometry'].apply(get_h3_cell)
# 这一步定义变量 h3_df_exploded
h3_df_exploded = gdf[['LocationID', 'borough', 'zone', 'centroid_h3_cells']].copy()


In [0]:
# 1. 将 Pandas DataFrame 转换为 Spark DataFrame
# 注意：转换时 Spark 会根据 Pandas 的数据自动推断 Schema
from pyspark.sql.window import Window 
from pyspark.sql.functions import row_number, col 

windowSpec = Window.partitionBy("VendorID","tpep_pickup_datetime", "tpep_dropoff_datetime").orderBy(col("tolls_amount").desc())

# 2. 增加行号并过滤
yellow_taxi_df = (
    yellow_taxi_raw_df
    .withColumn("row_num", row_number().over(windowSpec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

spark_h3_df = spark.createDataFrame(h3_df_exploded)


# 2. 执行 Join (现在两边都是 Spark DataFrame 了)
yellow_taxi_enriched_trips = yellow_taxi_df.join(
    spark_h3_df.select("LocationID", "centroid_h3_cells"), 
    on=yellow_taxi_df.PULocationID == spark_h3_df.LocationID, 
    how="inner"
)

# 3. 检查结果
# 注意：请确认字段名是 lpep_... 还是 tpep_...（根据你之前定义的 Schema 应该是 lpep）
yellow_taxi_enriched_trips.select("tpep_pickup_datetime", "PULocationID", "centroid_h3_cells").show(10)

In [0]:
display(left_df.limit(10))

In [0]:
from pyspark.sql.functions import expr, to_unix_timestamp 


left_df = yellow_taxi_enriched_kaggle_df.withColumn("pickupTimeUnix", to_unix_timestamp("pickupDatetime")).withColumn("dropoffTimeUnix", to_unix_timestamp("dropoffDatetime")).alias("left_df")

right_df = yellow_taxi_enriched_trips.withColumn("pickupTimeUnix", to_unix_timestamp("tpep_pickup_datetime")).withColumn("dropoffTimeUnix", to_unix_timestamp("tpep_dropoff_datetime")).alias("right_df")

distance_analysis = left_df.join(right_df.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "VendorID", "centroid_h3_cells"), on=expr("left_df.pickupTimeUnix = right_df.pickupTimeUnix and left_df.dropoffTimeUnix = right_df.dropoffTimeUnix and left_df.vendorID = right_df.VendorID"), how = "inner")



In [0]:
from pyspark.sql.functions import expr, to_unix_timestamp 


left_df = yellow_taxi_enriched_kaggle_df.withColumn("pickupTimeUnix", to_unix_timestamp("pickupDatetime")).withColumn("dropoffTimeUnix", to_unix_timestamp("dropoffDatetime"))

right_df = yellow_taxi_enriched_trips.withColumn("pickupTimeUnix", to_unix_timestamp("tpep_pickup_datetime")).withColumn("dropoffTimeUnix", to_unix_timestamp("tpep_dropoff_datetime"))

distance_analysis = left_df.alias("left_df").join(right_df.alias("right_df").select("tpep_pickup_datetime", "tpep_dropoff_datetime", "VendorID", "centroid_h3_cells"), on=expr("left_df.vendorID = right_df.VendorID"), how = "inner")



In [0]:
# 左表：转换完后起别名
L = yellow_taxi_enriched_kaggle_df \
    .withColumn("pickupTimeUnix", to_unix_timestamp("pickupDatetime")) \
    .withColumn("dropoffTimeUnix", to_unix_timestamp("dropoffDatetime")) \
    .alias("L")

# 右表：先转换、先 Select，最后起别名
R = yellow_taxi_enriched_trips \
    .withColumn("pickupTimeUnix", to_unix_timestamp("tpep_pickup_datetime")) \
    .withColumn("dropoffTimeUnix", to_unix_timestamp("tpep_dropoff_datetime")) \
    .select("pickupTimeUnix", "dropoffTimeUnix", "VendorID", "centroid_h3_cells") \
    .alias("R")

# 执行 Join，使用短别名 L 和 R，代码更清爽
distance_analysis = L.join(
    R, 
    on=expr("L.pickupTimeUnix = R.pickupTimeUnix AND L.dropoffTimeUnix = R.dropoffTimeUnix AND L.vendorID = R.VendorID"), 
    how="inner"
)

In [0]:
display(distance_analysis.limit(10))

In [0]:
distance_analysis.withColumn("YYYYMM", expr("year(dropoffDatetime)*100 + month(dropoffDatetime)")).groupBy("YYYYMM").count().orderBy("YYYYMM").display()


In [0]:
yellow_taxi_enriched_kaggle_df.withColumn("YYYYMM", expr("year(dropoffDatetime)*100 + month(dropoffDatetime)")).groupBy("YYYYMM").count().orderBy("YYYYMM").display()

In [0]:
display(distance_analysis.limit(10))

## How many can be matched

In [0]:
display(distance_analysis.limit(10))

In [0]:
from pyspark.sql import functions as F

# 假设 df 中同时拥有：
# 1. actual_h3 (根据经纬度算出来的)
# 2. centroid_h3 (根据 PULocationID 关联映射表得到的)

comparison_df = distance_analysis.withColumn(
    "is_match", 
    F.when(F.col("pickup_actual_h3") == F.col("centroid_h3_cells"), 1).otherwise(0)
)

# 计算整体匹配率
match_stats = comparison_df.select(F.avg("is_match")).collect()[0][0]
print(f"全局匹配准确率: {match_stats * 100:.2f}%")

In [0]:
import h3
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

# 定义计算格子距离的 UDF
@udf(returnType=IntegerType())
def get_grid_dist(h1, h2):
    try:
        return h3.grid_distance(h1, h2)
    except:
        return -1

dist_df = comparison_df.withColumn("dist_error", get_grid_dist("pickup_actual_h3", "centroid_h3_cells"))

window_spec = Window.partitionBy()
# 查看偏差分布：有多少订单偏差了 0 格、1 格、2 格...
error_stats = dist_df.groupBy("dist_error").count() \
    .withColumn("total", F.sum("count").over(window_spec)) \
    .withColumn("percentage_percent", (F.col("count") / F.col("total")) * 100) \
    .drop("total") \
    .orderBy("dist_error")

error_stats.select("dist_error", "count", F.concat(F.round("percentage_percent", 2), F.lit("%"))).show()


In [0]:
import matplotlib.pyplot as plt

# 转化为 Pandas 进行绘图
plot_df = dist_df.groupBy("dist_error").count().orderBy("dist_error").limit(5).toPandas()

plt.figure(figsize=(10, 6))
plt.bar(plot_df['dist_error'], plot_df['count'], color='crimson')
plt.title("H3 Grid Distance Error Distribution")
plt.xlabel("Distance (Number of Hexagons)")
plt.ylabel("Trip Count")
plt.grid(axis='y', linestyle='--', alpha=0.7)
display(plt.show())

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. 准备数据 (同上)
plot_pdf = dist_df.groupBy("dist_error").count().orderBy("dist_error").limit(6).toPandas()

# 2. 设置 Seaborn 主题
sns.set_theme(style="whitegrid")

# 3. 绘图
plt.figure(figsize=(12, 7))
ax = sns.barplot(x="dist_error", y="count", data=plot_pdf, palette="Reds_d")

# 4. 添加标题和标签
ax.set_title("H3 Grid Distance Error Distribution", fontsize=16)
ax.set_xlabel("Distance (Hexagons)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)

# 5. 在 Databricks 中显示
display(plt.show())

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. 准备数据（将 PySpark 转为 Pandas）
# 注意：我们取全部数据或大样本，不仅限前5个，以观察长尾
pdf = dist_df.select("dist_error").sample(0.1).toPandas() 

# 2. 绘图
plt.figure(figsize=(12, 6))

# 使用 sns.histplot 绘制直方图，并开启 kde=True 绘制平滑曲线
sns.histplot(pdf['dist_error'], kde=True, color="crimson", bins=30)

plt.title("H3 Grid Distance Error - Density Distribution", fontsize=16)
plt.xlabel("Distance Error (Hexagons)", fontsize=12)
plt.ylabel("Density / Frequency", fontsize=12)
plt.xlim(0, 5) # 限制 X 轴，因为 5 以后几乎没有数据了

display(plt.show())